# Data Quality — Delete Checks Notebook

**Use this notebook to remove checks from the registry.**

You can:
- **Soft delete** (set `is_active = false`) — the check data stays in the table but won't be validated
- **Hard delete** — permanently remove the row

Soft delete is safer for auditing; hard delete is cleaner if you don't need historical records.

In [ ]:
# ── Configuration ────────────────────────────────────────────────────
LAKEHOUSE_NAME = "MyLakehouse"    # Your existing Lakehouse name
SCHEMA_NAME    = "data_quality"   # Schema where tables were created

FULL_SCHEMA = f"{LAKEHOUSE_NAME}.{SCHEMA_NAME}"

## Current Checks

Here are all the checks currently registered:

In [ ]:
from IPython.display import display

display(spark.sql(f"""
SELECT check_name, model_name, is_active, run_frequency, workspace_name, dataset_name 
FROM {FULL_SCHEMA}.check_registry 
ORDER BY check_name, model_name
""").toPandas())

## Delete Checks

Choose how you want to delete:

1. **Soft Delete (recommended):** Set `is_active = false` — check stays in the table but won't be validated
2. **Hard Delete:** Permanently remove the row from the table

Edit the cell below to specify which checks to delete. You can match by:
- **check_name only** — deletes all models for that check
- **check_name + model_name** — deletes only that specific model for that check

In [ ]:
# ── EDIT BELOW: List the checks to delete ───────────────────────────────────
#
#   Format: (check_name, model_name_or_None)
#   - If model_name is None, deletes ALL models for that check_name
#   - If model_name is specified, deletes only that model for that check_name
#
to_delete = [
    ("Total Sales", "Sales EMEA"),        # Delete only Sales EMEA for Total Sales check
    ("Total Customers", None),             # Delete ALL models for Total Customers check
]

DELETE_METHOD = "soft"  # Choose: "soft" (set is_active=false) or "hard" (permanently delete)
# ─────────────────────────────────────────────────────────────────────────────

deleted_count = 0

if DELETE_METHOD == "soft":
    for check_name, model_name in to_delete:
        if model_name is None:
            # Soft delete all models for this check
            spark.sql(f"""
                UPDATE {FULL_SCHEMA}.check_registry
                SET is_active = false
                WHERE check_name = '{check_name}'
            """)
            count = spark.sql(f"SELECT COUNT(*) FROM {FULL_SCHEMA}.check_registry WHERE check_name = '{check_name}' AND is_active = false").collect()[0][0]
            deleted_count += count
            print(f"  ✓ Deactivated {count} row(s) for check: {check_name}")
        else:
            # Soft delete specific model for this check
            spark.sql(f"""
                UPDATE {FULL_SCHEMA}.check_registry
                SET is_active = false
                WHERE check_name = '{check_name}' AND model_name = '{model_name}'
            """)
            deleted_count += 1
            print(f"  ✓ Deactivated {check_name} → {model_name}")

elif DELETE_METHOD == "hard":
    for check_name, model_name in to_delete:
        if model_name is None:
            # Hard delete all models for this check
            spark.sql(f"""
                DELETE FROM {FULL_SCHEMA}.check_registry
                WHERE check_name = '{check_name}'
            """)
            print(f"  ✓ Deleted all rows for check: {check_name}")
        else:
            # Hard delete specific model for this check
            spark.sql(f"""
                DELETE FROM {FULL_SCHEMA}.check_registry
                WHERE check_name = '{check_name}' AND model_name = '{model_name}'
            """)
            print(f"  ✓ Deleted {check_name} → {model_name}")
        deleted_count += 1

print(f"\n✓  Deletion complete ({DELETE_METHOD} delete) — {deleted_count} operation(s)")

## Verify Remaining Checks

In [ ]:
from IPython.display import display

remaining = spark.sql(f"""
SELECT check_name, model_name, is_active, run_frequency, workspace_name, dataset_name 
FROM {FULL_SCHEMA}.check_registry 
WHERE is_active = true
ORDER BY check_name, model_name
""").toPandas()

if len(remaining) > 0:
    display(remaining)
    print(f"\n✓  {len(remaining)} active check(s) remain")
else:
    print("✓  No active checks remaining")